# Infra-Bench CLS — AlphaEarth Foundations Linear Probe

Linear-probe evaluation of AlphaEarth Foundations (2024) precomputed
embeddings on the Infra-Bench CLS 13-class benchmark. 3 seeds, spatial split.

## Model

- **AlphaEarth Foundations** (Google, 2024). Earth Engine collection
  `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`.
- Exposes **precomputed 64-dimensional embeddings** — not a trainable
  encoder. There is no "frozen vs fine-tuned" distinction; embeddings are
  always frozen by design. Only the linear head is trained.

## Input pipeline

### Feature source
64-D annual (2024) region-mean embeddings per tile. Embeddings were
fetched once via a companion Earth Engine job and cached to
`/.../data/alphaearth/embeddings_2024.parquet` (columns: `asset_id`,
`region`, `sector`, `asset_type`, `A00`..`A63`). This notebook re-uses that
parquet — no re-fetch.

### Normalization
None. Embeddings are consumed as-is.

## Split (shared across all Infra-Bench CLS FMs)

Spatial block-based split loaded from
`data/spatial_split/asset_id_to_split_v1.parquet`. Blocks assign a whole
~0.5° spatial region to the same partition so tiles near a training asset
cannot leak into val or test. The split is invariant across seeds and
across every FM in Infra-Bench CLS for fair cross-FM comparison.

## Training protocol

- **25 epochs**, batch 16, AdamW, LR **1e-3**
- Class-weighted CE, weights capped at 10×
- Head: `LinearProbeHead` — `Linear(64 → 13)` with `Dropout(0.1)`. ~845
  trainable parameters. Runs on CPU in ~15 min/seed; no GPU needed.
- **3 seeds** (314, 271, 161). Each seed varies the head init +
  DataLoader shuffle only; features are seed-invariant since embeddings
  are precomputed (linear-probe-on-frozen-features convention per Chen
  et al. 2020).
- **Best-val checkpoint restored before the held-out test pass**
  (search this notebook for `BEST_CKPT_BEFORE_TEST`).

## Per-sector F1 definition

Macro-average of per-class F1s for the classes in that sector, computed
on the FULL test set (not filtered to in-sector samples). Return schema:
`{n, macro_f1, acc, per_class_f1_in_sector}` per sector — matches every
other FM in Infra-Bench CLS for cross-FM aggregation.

## Aggregate output

Combines the 3 seeds into `mean ± std` and `per_seed` arrays for every
metric. Write is gated by `set(SEEDS) == set(FULL_PROTOCOL_SEEDS)` —
partial reruns will not overwrite an existing 3-seed aggregate.

## Outputs

- Per-seed: `results/fm_eval_alphaearth_v2_spatial/alphaearth_v2_seed{314,271,161}_results.json`
- Aggregate: `results/fm_eval_alphaearth_v2_spatial/alphaearth_v2_aggregate.json`
- Confusion matrix: `results/fm_eval_alphaearth_v2_spatial/confusion_matrix_alphaearth_v2_aggregate.png`

## Runtime

`SMOKE_ONLY=True` by default. Set to `False` and re-run the training cell
to launch the full 3-seed run.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}  '
      f'(not required — 64-D linear probe runs fine on CPU)')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: False  (not required — 64-D linear probe runs fine on CPU)


In [2]:
%%capture
!pip install -q pyarrow pyproj scikit-learn


In [3]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Only load_split_artifact is needed at training time. If the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
Could not import (zip is pre-Phase-1): No module named 'curation.utils.spatial_blocking'. Using inline fallback.


In [4]:
import os
from pathlib import Path

DATASETS_DRIVE       = f'{DRIVE_ROOT}/datasets'
ALPHAEARTH_DATA_DIR  = f'{DRIVE_ROOT}/data/alphaearth'
EMBEDDINGS_PARQUET   = f'{ALPHAEARTH_DATA_DIR}/embeddings_2024.parquet'
SPLIT_ARTIFACT_PATH  = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
OUTPUT_DIR           = f'{DRIVE_ROOT}/results/fm_eval_alphaearth_v2_spatial'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Coverage --------------------------------------------------------------
REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

# ---- 13-class taxonomy (locked, must match all FM notebooks) ----
CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX        = {n: i for i, n in enumerate(CLASS_NAMES)}
CLASS_IDX_TO_SECTOR = {i: n.split('.')[0] for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':              'water.water_works',   # legacy manifest tag
    'water.water_works':                  'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

# ---- AlphaEarth-specific ---------------------------------------------------
AE_BAND_COUNT = 64
AE_BAND_NAMES = [f'A{i:02d}' for i in range(AE_BAND_COUNT)]

# ---- Per-sector partition (v2 definition — same as CROMA v2) --------------
SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# ---- Training (mirrors CROMA v2 / Prithvi / SatlasS1) ---------------------
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0
SEEDS      = [314, 271, 161]  # pi*100, e*100, phi*100

def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f'Embeddings parquet:  {EMBEDDINGS_PARQUET}')
print(f'Split artifact:      {SPLIT_ARTIFACT_PATH}')
print(f'Output dir:          {OUTPUT_DIR}')
print(f'Training seeds:      {SEEDS}')
print(f'Linear probe:        {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')


Embeddings parquet:  /content/drive/MyDrive/infra_fm/data/alphaearth/embeddings_2024.parquet
Split artifact:      /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
Output dir:          /content/drive/MyDrive/infra_fm/results/fm_eval_alphaearth_v2_spatial
Training seeds:      [314, 271, 161]
Linear probe:        25 epochs, batch 16, lr 0.001


In [5]:
import pandas as pd
import numpy as np

embeddings_df = pd.read_parquet(EMBEDDINGS_PARQUET)
print(f'Parquet shape: {embeddings_df.shape}  '
      f'(expect 4 metadata cols + {AE_BAND_COUNT} feature cols)')

# Sanity: all expected band columns present.
missing_band_cols = [c for c in AE_BAND_NAMES if c not in embeddings_df.columns]
if missing_band_cols:
    raise RuntimeError(f'Embeddings parquet missing band columns: {missing_band_cols[:5]}...')

# Defensive NaN check (Phase 1 verification dropped AE-missing tiles
# already; if any NaNs reach here, surface them but don't crash).
emb_arr = embeddings_df[AE_BAND_NAMES].to_numpy(dtype=np.float64)
finite_mask = np.isfinite(emb_arr).all(axis=1)
n_bad = (~finite_mask).sum()
if n_bad > 0:
    print(f'WARNING: {n_bad} rows have non-finite embeddings; dropping them.')
    print(f'  first 5 bad asset_ids: {embeddings_df.loc[~finite_mask, "asset_id"].head().tolist()}')
    embeddings_df = embeddings_df[finite_mask].reset_index(drop=True)

print(f'\nEmbedding statistics (all 64 bands, all rows):')
print(f'  min:  {emb_arr[finite_mask].min():.4f}')
print(f'  max:  {emb_arr[finite_mask].max():.4f}')
print(f'  mean: {emb_arr[finite_mask].mean():.4f}')
print(f'  std:  {emb_arr[finite_mask].std():.4f}')

# Map asset_type -> integer label; drop any unrecognized class
n_before = len(embeddings_df)
# water_works legacy translation: map raw manifest tag 'water.treatment.plant'
# to canonical 'water.water_works' BEFORE the isin(CLASS_NAMES) filter, so legacy
# rows aren't dropped. Also covers any other legacy tags in ASSET_TYPE_MAP.
embeddings_df['asset_type'] = embeddings_df['asset_type'].map(
    lambda x: ASSET_TYPE_MAP.get(x, x)
)
embeddings_df = embeddings_df[embeddings_df['asset_type'].isin(CLASS_NAMES)].reset_index(drop=True)
n_dropped_label = n_before - len(embeddings_df)
if n_dropped_label:
    print(f'\nDropped {n_dropped_label} rows with asset_type not in CLASS_NAMES.')
embeddings_df['label'] = embeddings_df['asset_type'].map(CLASS_TO_IDX).astype(int)

print(f'\nFinal embedding table: {len(embeddings_df):,} rows')
print('\nPer-region:')
print(embeddings_df.groupby('region').size().to_string())
print('\nPer-class:')
print(embeddings_df.groupby('asset_type').size().sort_values(ascending=False).to_string())


Parquet shape: (18750, 68)  (expect 4 metadata cols + 64 feature cols)
  first 5 bad asset_ids: ['osm_way_1036487691']

Embedding statistics (all 64 bands, all rows):
  min:  -0.4462
  max:  0.4516
  mean: -0.0075
  std:  0.1135

Final embedding table: 18,749 rows

Per-region:
region
africa               2842
asia                 2765
australia-oceania    3014
central-america      2565
europe               1736
north-america        2823
south-america        3004

Per-class:
asset_type
transport.train_station           5046
water.storage_tank                3993
energy.distribution.other         3264
water.wastewater.plant            1299
energy.distribution.substation     975
water.treatment.plant              895
transport.airport                  871
energy.transmission.substation     676
energy.generation.solar_farm       631
energy.generation.power_plant      545
telecom.data_center                532
energy.generation.wind_farm         11
transport.port_terminal             11


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Load the spatial split parquet and join on asset_id.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  split distribution: {Counter(asset_to_split.values())}')

embeddings_df['asset_id'] = embeddings_df['asset_id'].astype(str)
embeddings_df['split']    = embeddings_df['asset_id'].map(asset_to_split)

# Defensive coverage check: every embedding row should land in a split
# because Phase 1 dropped AE-missing tiles before producing the artifact.
n_unmapped = embeddings_df['split'].isna().sum()
if n_unmapped:
    print(f'WARNING: {n_unmapped} embedding rows have no spatial-split assignment. '
          f'Dropping them; investigate if non-zero.')
    print(f'  first 5 unmapped asset_ids: {embeddings_df.loc[embeddings_df["split"].isna(), "asset_id"].head().tolist()}')
    embeddings_df = embeddings_df[embeddings_df['split'].notna()].reset_index(drop=True)
else:
    print(f'Full coverage: every embedding row has a split assignment.')


class AEEmbeddingDataset(Dataset):
    """Tensor of 64-D AlphaEarth embeddings + metadata for one row at a time.

    Built once per (split). Holds an in-memory (N, 64) float32 matrix so
    DataLoader workers aren't needed.
    """
    def __init__(self, df, indices):
        self.df = df.iloc[indices].reset_index(drop=True)
        self._emb_mat = self.df[AE_BAND_NAMES].to_numpy(dtype=np.float32)
        self.labels   = self.df['label'].astype(int).tolist()
        self.regions  = self.df['region'].tolist()
        self.sectors  = self.df['sector'].tolist()
        self.asset_ids = self.df['asset_id'].astype(str).tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        return {
            'embedding': torch.from_numpy(self._emb_mat[i]),
            'label':     int(self.labels[i]),
            'region':    self.regions[i],
            'sector':    self.sectors[i],
            'asset_id':  self.asset_ids[i],
        }


train_idx = embeddings_df.index[embeddings_df['split'] == 'train'].tolist()
val_idx   = embeddings_df.index[embeddings_df['split'] == 'val'].tolist()
test_idx  = embeddings_df.index[embeddings_df['split'] == 'test'].tolist()

train_set = AEEmbeddingDataset(embeddings_df, train_idx)
val_set   = AEEmbeddingDataset(embeddings_df, val_idx)
test_set  = AEEmbeddingDataset(embeddings_df, test_idx)

print(f'\nSplits: train={len(train_set):,}  val={len(val_set):,}  test={len(test_set):,}')
print(f'  total: {len(train_set) + len(val_set) + len(test_set):,}')

print('\nClass distribution per split:')
print(f'  {"class":<35s} {"train":>6s} {"val":>6s} {"test":>6s}')
for c, name in enumerate(CLASS_NAMES):
    tr = sum(1 for l in train_set.labels if l == c)
    va = sum(1 for l in val_set.labels   if l == c)
    te = sum(1 for l in test_set.labels  if l == c)
    print(f'  [{c:>2d}] {name:<30s} {tr:>6d} {va:>6d} {te:>6d}')


Loaded split artifact: 18,750 asset_id -> split entries
  split distribution: Counter({'train': 13087, 'val': 2851, 'test': 2812})
Full coverage: every embedding row has a split assignment.

Splits: train=13,086  val=2,851  test=2,812
  total: 18,749

Class distribution per split:
  class                                train    val   test
  [ 0] energy.transmission.substation    459    112    105
  [ 1] energy.distribution.substation    671    163    141
  [ 2] energy.distribution.other        2314    439    511
  [ 3] energy.generation.power_plant     366    109     70
  [ 4] energy.generation.solar_farm      432    101     98
  [ 5] energy.generation.wind_farm         7      1      3
  [ 6] water.wastewater.plant            837    234    228
  [ 7] water.treatment.plant             659    118    118
  [ 8] water.storage_tank               2801    619    573
  [ 9] transport.airport                 589    157    125
  [10] transport.train_station          3559    750    737
  [11] tra

In [7]:
# Diagnostic: regenerate the AE v1 random stratified split inside this
# notebook for exact-match comparison against the new spatial split.
# Same logic as v1 AE cell 9: stratified by class, 70/15/15, seed=42.
import random

def old_stratified_split(df, train_frac=0.7, val_frac=0.15, seed=42):
    """Exact mirror of v1 AlphaEarth's stratified_split on a DataFrame."""
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(df['label'].unique()):
        idxs = df.index[df['label'] == cls].tolist()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


# IMPORTANT: regenerate against the FILTERED embeddings_df (post-split-
# coverage check + post-class-filter), not the raw parquet. This matches
# the operational order: AE v1 also filtered to in-CLASS_NAMES before
# splitting.
tr_old, va_old, te_old = old_stratified_split(embeddings_df)
old_split = {}
for i in tr_old: old_split[embeddings_df.iloc[i]['asset_id']] = 'train'
for i in va_old: old_split[embeddings_df.iloc[i]['asset_id']] = 'val'
for i in te_old: old_split[embeddings_df.iloc[i]['asset_id']] = 'test'

# Compare against the spatial split.
new_split = dict(zip(embeddings_df['asset_id'], embeddings_df['split']))
common_ids = set(old_split) & set(new_split)
print('=' * 76)
print(f'Diagnostic: old (random stratified) vs new (spatial) split '
      f'({len(common_ids):,} tiles)')
print('=' * 76)
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], new_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Should be close to CROMA v2\'s ~47% changed — the spatial split '
      'is the same; only the training data layer differs across FMs.)')


Diagnostic: old (random stratified) vs new (spatial) split (18,749 tiles)

old \ new      train       val       test
--------------------------------------------------
train           9200      1984       1934
val             1908       429        467
test            1978       438        411

Unchanged: 10,040 (53.5%)
Changed:   8,709 (46.5%)

(Should be close to CROMA v2's ~47% changed — the spatial split is the same; only the training data layer differs across FMs.)


In [8]:
import torch.nn as nn
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  (linear probe on 64-D embeddings — CPU is fine)')


class LinearProbeHead(nn.Module):
    """Single Linear: 64 -> 13. No backbone — AlphaEarth embeddings are
    precomputed and frozen by design. backbone.NAME is 'alphaearth_v1'
    (the embedding set is unchanged; only the eval protocol differs)."""
    NAME = 'alphaearth_v1'

    def __init__(self, in_dim=AE_BAND_COUNT, num_classes=len(CLASS_NAMES),
                 dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, num_classes),
        )
    def forward(self, x):
        return self.head(x)


def compute_class_weights(labels, max_weight=WEIGHT_CAP):
    """List-of-int-labels signature (kept from AE v1 since the embedding
    dataset doesn't have CROMA's ConcatDataset / SubsetView hierarchy)."""
    counts = Counter(labels)
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'embedding': torch.stack([b['embedding'] for b in batch]),
        'label':     torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region':    [b['region'] for b in batch],
        'sector':    [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """Per-sector F1 v2 — matches CROMA v2's `_per_sector_v2` exactly.
    Macro-average of per-class F1s for the classes in that sector,
    computed on the FULL test set. Returns dict with 'n', 'macro_f1',
    'acc', 'per_class_f1_in_sector' per sector.

    Note: AlphaEarth v1 already used this definition (under the name
    `per_sector_corrected`). This implementation normalizes the return
    shape to CROMA v2's schema for cross-FM compatibility."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['embedding'].to(DEVICE, non_blocking=False))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        # per_region unchanged (each tile has a unique region label).
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition). '
            'AlphaEarth v1 already used this definition (under the name '
            'per_sector_corrected); this notebook normalizes the return '
            'shape to CROMA v2\'s schema for cross-FM compatibility.'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set):
    set_seed(seed)
    print(f'\n--- seed {seed} ---')
    head = LinearProbeHead().to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    # CPU runtime: num_workers=0 (data in memory) + pin_memory=False.
    train_loader = DataLoader(train_set, batch_size=LP_BATCH, shuffle=True,
                              num_workers=0, collate_fn=collate,
                              pin_memory=False, generator=g)
    val_loader   = DataLoader(val_set,   batch_size=LP_BATCH, shuffle=False,
                              num_workers=0, collate_fn=collate, pin_memory=False)
    test_loader  = DataLoader(test_set,  batch_size=LP_BATCH, shuffle=False,
                              num_workers=0, collate_fn=collate, pin_memory=False)

    weights = compute_class_weights(train_set.labels).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(head.parameters(), lr=LP_LR, weight_decay=1e-4)

    run_name = f'alphaearth_v2_seed{seed}_linear_probe'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt  = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt = ckpt_dir / 'checkpoint_final.pt'

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(LP_EPOCHS):
        head.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            emb    = batch['embedding'].to(DEVICE, non_blocking=False)
            labels = batch['label'].to(DEVICE, non_blocking=False)
            optimizer.zero_grad()
            loss = criterion(head(emb), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * emb.size(0)
            n += emb.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(head, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })
        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch  = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': head.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': LP_EPOCHS,
                'model_state_dict': head.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST =================================
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        head.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(head, test_loader, return_breakdowns=True)
    return {
        'run_name':     run_name,
        'backbone':     head.NAME,
        'condition':    'precomputed_embeddings_linear_probe',
        'num_epochs':   LP_EPOCHS,
        'seed':         seed,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).')


Device: cpu  (linear probe on 64-D embeddings — CPU is fine)
Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).


In [10]:
# ============================================================================
# SMOKE CHECK — load probe, build one batch, run forward + one train step.
# Default SMOKE_ONLY=True so the multi-seed cell below exits before training.
# ============================================================================
SMOKE_ONLY = False

set_seed(SEEDS[0])
head = LinearProbeHead().to(DEVICE)
n_params = sum(p.numel() for p in head.parameters())
print(f'Linear probe params: {n_params:,}  '
      f'(expect 64*13 + 13 = {AE_BAND_COUNT * len(CLASS_NAMES) + len(CLASS_NAMES)})')

smoke_loader = DataLoader(train_set, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate, pin_memory=False)
batch = next(iter(smoke_loader))
emb = batch['embedding']
print(f'  Batch embedding shape: {tuple(emb.shape)}  (expect [4, {AE_BAND_COUNT}])')
print(f'  Batch embedding dtype: {emb.dtype}')
print(f'  Batch embedding range: [{emb.min().item():.4f}, {emb.max().item():.4f}]')
print(f'  Batch labels:          {batch["label"].tolist()}')

with torch.no_grad():
    logits = head(emb.to(DEVICE))
print(f'  Logits shape:          {tuple(logits.shape)}  (expect [4, {len(CLASS_NAMES)}])')
print(f'  Logits range:          [{logits.min().item():.4f}, {logits.max().item():.4f}]')

finite = torch.isfinite(logits).all().item()
print(f'  All finite:            {finite}')
assert finite, 'NaN/Inf in logits — aborting smoke check'

# One train step
weights = compute_class_weights(train_set.labels).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(head.parameters(), lr=LP_LR, weight_decay=1e-4)
optimizer.zero_grad()
loss = criterion(head(emb.to(DEVICE)), batch['label'].to(DEVICE))
loss.backward()
head_grads = [p.grad for p in head.head.parameters() if p.grad is not None]
assert head_grads and any(g.abs().sum().item() > 0 for g in head_grads), \
    'head received zero gradients'
optimizer.step()
print(f'  Train step loss:       {loss.item():.4f}  (finite={torch.isfinite(loss).item()})')

print(f'\nSmoke check PASSED. SMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
      f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


Linear probe params: 845  (expect 64*13 + 13 = 845)
  Batch embedding shape: (4, 64)  (expect [4, 64])
  Batch embedding dtype: torch.float32
  Batch embedding range: [-0.3163, 0.2936]
  Batch labels:          [1, 1, 1, 1]
  Logits shape:          (4, 13)  (expect [4, 13])
  Logits range:          [-0.1867, 0.2068]
  All finite:            True
  Train step loss:       2.6273  (finite=True)

Smoke check PASSED. SMOKE_ONLY = False — multi-seed cell below will run all 3 seeds.


In [11]:
# ============================================================================
# Multi-seed invocation. Same shape as CROMA v2: per-seed JSON + aggregate
# JSON (mean / std / per_seed arrays) + summed-then-row-normalized
# confusion PNG. Gated by SMOKE_ONLY from the cell above.
# ============================================================================
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping all training. Set False in the cell above '
          'and re-run that cell + this one to train.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        result = train_one_seed(seed,
                                train_set=train_set,
                                val_set=val_set,
                                test_set=test_set)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'alphaearth_v2_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'linear_probe': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    # ---- Aggregate stats with per_seed arrays ---------------------------
    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {
            'mean':     float(arr.mean()),
            'std':      float(arr.std(ddof=0)),
            'per_seed': [float(v) for v in arr],
        }

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {
            'class':    CLASS_NAMES[i],
            'idx':      i,
            'mean_f1':  float(per_class_arr[:, i].mean()),
            'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
            'per_seed': [float(v) for v in per_class_arr[:, i]],
        }
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s_per_seed = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1']
                        for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s_per_seed)),
            'std_macro_f1':  float(np.std(f1s_per_seed, ddof=0)),
            'per_seed':      [float(v) for v in f1s_per_seed],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s_per_seed = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
                        for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s_per_seed)),
            'std_macro_f1':  float(np.nanstd(f1s_per_seed, ddof=0)),
            'per_seed':      [float(v) for v in f1s_per_seed],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    agg_path = Path(OUTPUT_DIR) / 'alphaearth_v2_aggregate.json'
    with open(agg_path, 'w') as f:
        _json.dump(agg, f, indent=2)
    print(f'\nAggregate saved: {agg_path}')



--- seed 314 ---
  ep   1  loss=2.3834  val_acc=0.3101  val_f1=0.1630 *
  ep   2  loss=2.2068  val_acc=0.2852  val_f1=0.1790 *
  ep   3  loss=2.1167  val_acc=0.2876  val_f1=0.1895 *
  ep   4  loss=2.0597  val_acc=0.2869  val_f1=0.1882
  ep   5  loss=2.0217  val_acc=0.2838  val_f1=0.1909 *
  ep   6  loss=1.9941  val_acc=0.2845  val_f1=0.1955 *
  ep   7  loss=1.9677  val_acc=0.2774  val_f1=0.1904
  ep   8  loss=1.9472  val_acc=0.2894  val_f1=0.2011 *
  ep   9  loss=1.9367  val_acc=0.2796  val_f1=0.1955
  ep  10  loss=1.9202  val_acc=0.2876  val_f1=0.2012 *
  ep  11  loss=1.9100  val_acc=0.2936  val_f1=0.2035 *
  ep  12  loss=1.8994  val_acc=0.2925  val_f1=0.2037 *
  ep  13  loss=1.8971  val_acc=0.2908  val_f1=0.2049 *
  ep  14  loss=1.8876  val_acc=0.2950  val_f1=0.2076 *
  ep  15  loss=1.8805  val_acc=0.2953  val_f1=0.2086 *
  ep  16  loss=1.8676  val_acc=0.2901  val_f1=0.2063
  ep  17  loss=1.8667  val_acc=0.2946  val_f1=0.2101 *
  ep  18  loss=1.8610  val_acc=0.2974  val_f1=0.2116 *


In [12]:
# Confusion-matrix PNG (sum across seeds, row-normalized) + summary print.
# Runs only when training completed.
if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping confusion-matrix render and summary.')
else:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm),
                         where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap='Greens', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'AlphaEarth v2 spatial — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / 'confusion_matrix_alphaearth_v2_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    # ---- Summary -----------------------------------------------------
    print('\n' + '=' * 76)
    print(f'AlphaEarth v2 spatial — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- '
          f'{agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- '
          f'{agg["test_accuracy"]["std"]:.4f}')

    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')

    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')

    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')


Confusion matrix saved: /content/drive/MyDrive/infra_fm/results/fm_eval_alphaearth_v2_spatial/confusion_matrix_alphaearth_v2_aggregate.png

AlphaEarth v2 spatial — aggregate (3 seeds)
Test macro F1: 0.2223 +/- 0.0126
Test accuracy: 0.2690 +/- 0.0009

Per-class F1 (mean +/- std):
  [ 0] energy.transmission.substation     0.1446 +/- 0.0059
  [ 1] energy.distribution.substation     0.2058 +/- 0.0024
  [ 2] energy.distribution.other          0.1040 +/- 0.0013
  [ 3] energy.generation.power_plant      0.1832 +/- 0.0021
  [ 4] energy.generation.solar_farm       0.2330 +/- 0.0061
  [ 5] energy.generation.wind_farm        0.0000 +/- 0.0000
  [ 6] water.wastewater.plant             0.2589 +/- 0.0068
  [ 7] water.treatment.plant              0.1141 +/- 0.0020
  [ 8] water.storage_tank                 0.2460 +/- 0.0036
  [ 9] transport.airport                  0.4316 +/- 0.0024
  [10] transport.train_station            0.3894 +/- 0.0056
  [11] transport.port_terminal            0.1111 +/- 0.1571
